In [1]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import os
import numpy as np
from tensorflow.keras import layers
from IPython.display import Audio
import wave
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

2024-10-27 21:09:27.424474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-27 21:09:27.435331: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-27 21:09:27.438811: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-27 21:09:27.447786: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-27 21:09:28.011726: W tensorflow/compiler/tf2

In [2]:
df = pd.read_csv("../data_processing/sep28k-mfcc.csv")

In [3]:
df = df[df['NaturalPause'] == 0]
df = df[df['Block'] == 0]
df = df[df['Prolongation'] == 0]
df = df[df['WordRep'] == 0]
df = df[df['SoundRep'] == 0]
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
2,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
4,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
6,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
9,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
10,HeStutters,0,10,41861760,41909760,0,0,0,0,0,...,-2.898894,-6.209611,-3.659979,-5.923317,-2.773856,-3.547750,2.452112,1.432807,3.639793,2.238830


In [4]:
df = df.reset_index()
df.head()

,index,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,...,29,30,31,32,33,34,35,36,37,38
0,2,HeStutters,0,2,34809760,34857760,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,4,HeStutters,0,4,35721920,35769920,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,6,HeStutters,0,6,37251200,37299200,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,9,HeStutters,0,9,41417440,41465440,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,10,HeStutters,0,10,41861760,41909760,0,0,0,0,...,-2.898894,-6.209611,-3.659979,-5.923317,-2.773856,-3.547750,2.452112,1.432807,3.639793,2.238830


In [5]:
df = df.drop(columns=['index'])
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
0,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,HeStutters,0,10,41861760,41909760,0,0,0,0,0,...,-2.898894,-6.209611,-3.659979,-5.923317,-2.773856,-3.547750,2.452112,1.432807,3.639793,2.238830


In [6]:
df.to_csv("interjection.csv",index=False)

In [7]:
import os

def list_files(directory):
    return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

# Replace 'directory_path' with the path to your directory
directory_path = '/home/alien/Git/DATA/mel_spects_interjection'
# names_list = list_files(directory_path)
names_list = pd.read_csv("interjection.csv")['Name'].values.tolist()
# full_names_list = ["/home/alien/Git/DATA/mfcc_images/" + img + ".jpg" for img in names_list]
full_names_list = []
for img in names_list:
    corresponding_sound = df.loc[df['Name'] == img, 'Interjection'].values[0]
    if corresponding_sound == 0:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_interjection/" + img + "_fluent.jpg")
    if corresponding_sound >= 1:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_interjection/" + img + "_stutter.jpg")

# Replace 'directory_path' with the path to your directory
directory_path = '/home/alien/Git/DATA/mel_specaugment_interjection'
# names_list = list_files(directory_path)
names_list = pd.read_csv("interjection.csv")['Name'].values.tolist()
# full_names_list = ["/home/alien/Git/DATA/mfcc_images/" + img + ".jpg" for img in names_list]
for img in names_list:
    corresponding_sound = df.loc[df['Name'] == img, 'Interjection'].values[0]
    if corresponding_sound == 0:
        full_names_list.append("/home/alien/Git/DATA/mel_specaugment_interjection/" + img + "_fluent.jpg")
    if corresponding_sound >= 1:
        full_names_list.append("/home/alien/Git/DATA/mel_specaugment_interjection/" + img + "_stutter.jpg")

print(full_names_list[30])
print(full_names_list[-30])

/home/alien/Git/DATA/mel_spects_interjection/HeStutters_2_118_fluent.jpg
/home/alien/Git/DATA/mel_specaugment_interjection/WomenWhoStutter_106_7_fluent.jpg


In [8]:
def load_all(imagefile_list):
    data = []
    labels = []

    for imagefile in imagefile_list:
        print(imagefile)
        image = cv2.imread(imagefile)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        data.append(image)

        if "fluent" in imagefile:
            labels.append(0)
        else:
            labels.append(1)

    labels = np.array(labels)
    data = np.array(data)

    return data, labels

In [9]:
X, y = load_all(full_names_list)

/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_2_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_4_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_6_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_9_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_10_stutter.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_11_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_12_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_15_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_16_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_20_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_22_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_24_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_26_fluent.jpg
/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_29_fluent

In [10]:
print(X[0])
print(y[0])

[[[ 68  33  60]
  [ 96  28  68]
  [ 89  27  67]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[114  57  88]
  [166  57 117]
  [157  55 112]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[140  60  94]
  [190  55 118]
  [198  65 126]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 ...

 [[126  65 103]
  [194  56 123]
  [189  69 128]
  ...
  [ 29  13  72]
  [ 18   5  59]
  [ 21   6  57]]

 [[ 90  48 102]
  [145  31 116]
  [141  39 122]
  ...
  [ 26  18  68]
  [ 24  17  63]
  [ 27  19  61]]

 [[ 86  33  88]
  [122  36 120]
  [134  36 131]
  ...
  [ 23  14  63]
  [ 27  18  70]
  [ 26  14  66]]]
0


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.15, random_state=8)

In [12]:
print("x_train shape:", X_train.shape)
print("x_test shape:", X_test.shape)
print('y_train shape:', y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (4112, 224, 224, 3)
x_test shape: (726, 224, 224, 3)
y_train shape: (4112,)
y_test shape: (726,)


In [13]:
print(len(X_train), len(X_test), len(y_train), len(y_test))

4112 726 4112 726


In [14]:
from collections import Counter
print(Counter(y_train))
print(Counter(y_test))

Counter({0: 2883, 1: 1229})
Counter({0: 513, 1: 213})


In [15]:
print(X_train)

[[[[  0   0   0]
   [  0   0   0]
   [  0   0   0]
   ...
   [  5   4  24]
   [  6   4  19]
   [  7   4  23]]

  [[  0   0   0]
   [  0   0   1]
   [  0   0   2]
   ...
   [ 23  12  47]
   [ 23  17  41]
   [  3   1  22]]

  [[  0   0   3]
   [  0   0   4]
   [  0   0   4]
   ...
   [ 23   4  57]
   [ 30  17  57]
   [  4   2  27]]

  ...

  [[ 15   5  42]
   [ 47  22  89]
   [ 55  19 110]
   ...
   [ 98  19 128]
   [138  39 131]
   [133  46 138]]

  [[  8   4  18]
   [ 20  11  38]
   [ 24  10  48]
   ...
   [ 44  11  58]
   [ 62  20  58]
   [ 59  23  60]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   2]
   [  0   0   2]
   [  0   0   2]]]


 [[[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0

In [16]:
from PIL import Image
IMAGE_DIR = "/home/alien/Git/DATA/mel_spects_interjection/"

def get_image_dimensions(image_path):
    with Image.open(image_path) as img:
        width, height = img.size
    return width, height

image_path = '/home/alien/Git/DATA/mel_spects_interjection/HeStutters_0_0_fluent.jpg'  # Change this to the path of your image file
width, height = get_image_dimensions(image_path)
print("Image width:", width)
print("Image height:", height)

Image width: 610
Image height: 450


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, MultiHeadAttention
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dropout, Flatten, Dense, Input, AveragePooling2D, Attention, Reshape, TimeDistributed, Bidirectional, LSTM, GRU
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input, decode_predictions
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import ConvLSTM2D
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.constraints import ClipValue
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Layer, MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization

In [18]:
# def build_model(input_shape=(224, 224, 3)):
#     base_model = VGG19(weights='imagenet', include_top=True, input_tensor=Input(shape=input_shape))
#     # Get the output of the 'fc2' layer in VGG16
#     a = base_model.get_layer('fc2').output

#     # Add a Dense layer with 13 neurons
#     dense_layer = Dense(13, activation='relu')(a)

#     # Add a final output layer with sigmoid activation
#     output_layer = Dense(1, activation='sigmoid')(dense_layer)

#     # Define the model with VGG16 base and the added layers
#     model = Model(inputs=base_model.input, outputs=output_layer)

#     # Freeze the weights of the VGG16 layers
#     for layer in base_model.layers:
#         layer.trainable = False

#     return model

def build_model(input_shape=(224, 224, 3), use_float16=False):
    inputs = Input(shape=input_shape)

    # Convert to lower precision if specified
    if use_float16:
        x = tf.keras.layers.Lambda(lambda t: tf.cast(t, tf.float16))(inputs)
    else:
        x = inputs

    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = TimeDistributed(Flatten())(x)  # Flatten along the time dimension
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Attention()([x, x])  # Self-attention mechanism

    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    return model



In [19]:
  # base_model = VGG19(weights='imagenet', include_top=True,
  #                   input_tensor=Input(shape=(224, 224, 3)))
  # base_model.summary()

In [20]:
vgg_model = build_model()
vgg_model.summary()

I0000 00:00:1730077775.191746   35758 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730077775.213041   35758 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730077775.218873   35758 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730077775.232603   35758 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 112, 112,  │        256 │ max_pooling2d[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,856 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ max_pooling2d_1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    295,168 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │      1,024 │ max_pooling2d_2[… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 28, 7168)  │          0 │ batch_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 28, 256)   │  7,472,128 │ time_distributed… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 28, 256)   │          0 │ bidirectional[0]… │
│ (Attention)         │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 7168)      │          0 │ attention[0][0] 

 Total params: 12,355,649 (47.13 MB)

 Trainable params: 12,354,753 (47.13 MB)

 Non-trainable params: 896 (3.50 KB)

In [21]:
from tensorflow.keras import optimizers
vgg_model.compile(
  optimizer=Adam(0.001),
  loss='binary_crossentropy',
  metrics=['accuracy']
)

In [22]:
# batch_size = 16
# history = vgg_model.fit(
#     X_train,
#     y_train,
#     batch_size=batch_size,
#     epochs=35,
#     callbacks=[
#         ReduceLROnPlateau(
#             monitor = 'accuracy',
#             factor = 0.2,
#             patience = 5,
#             verbose = 1,
#             min_lr = 0.0001
#         ),
#         EarlyStopping(
#             monitor = 'accuracy',
#             patience = 10,
#             verbose = 1,
#             restore_best_weights = True
#         )
#     ]
# )
batch_size = 36
history = vgg_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=70,
    callbacks=[
        ReduceLROnPlateau(
            monitor = 'accuracy',
            patience = 5,
            verbose = 1,
            min_lr = 0.0001
        ),
    ]
)

Epoch 1/70


2024-10-27 21:09:39.358058: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907
W0000 00:00:1730077779.428221   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.450877   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.452879   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.454279   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.458965   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.467407   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.471467   35904 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077779.473243   35904 gpu_t

  1/115 ━━━━━━━━━━━━━━━━━━━━ 16:27 9s/step - accuracy: 0.6944 - loss: 0.6058

W0000 00:00:1730077785.026565   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.029135   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.032225   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.034942   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.038241   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.042319   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.045368   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.048648   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077785.052322   35902 gp

114/115 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.6430 - loss: 0.7497

W0000 00:00:1730077799.204142   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.205188   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.205982   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.206545   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.207214   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.207827   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.208481   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.209091   35902 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077799.209972   35902 gp

115/115 ━━━━━━━━━━━━━━━━━━━━ 24s 135ms/step - accuracy: 0.6431 - loss: 0.7490 - learning_rate: 0.0010
Epoch 2/70


W0000 00:00:1730077800.416563   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.424492   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.428311   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.434509   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.441190   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.448583   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.459496   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.476768   35899 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730077800.477696   35899 gp

115/115 ━━━━━━━━━━━━━━━━━━━━ 14s 124ms/step - accuracy: 0.6771 - loss: 0.6353 - learning_rate: 0.0010
Epoch 3/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.6970 - loss: 0.6088 - learning_rate: 0.0010
Epoch 4/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.7188 - loss: 0.5630 - learning_rate: 0.0010
Epoch 5/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.7063 - loss: 0.5677 - learning_rate: 0.0010
Epoch 6/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.7105 - loss: 0.5618 - learning_rate: 0.0010
Epoch 7/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 127ms/step - accuracy: 0.7508 - loss: 0.5070 - learning_rate: 0.0010
Epoch 8/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.7691 - loss: 0.4876 - learning_rate: 0.0010
Epoch 9/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 126ms/step - accuracy: 0.8038 - loss: 0.4387 - learning_rate: 0.0010
Epoch 10/70
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 127ms/step - accuracy: 0.8222 - loss: 0.4081 - learning_rate: 

In [25]:
# predictions
vgg_pred = vgg_model.predict(X_test, batch_size=1)

vgg_pred = np.round(vgg_pred)

  # model evaluation
confusion = confusion_matrix(y_test, vgg_pred)
print(classification_report(y_test, vgg_pred))
print(confusion)

726/726 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
              precision    recall  f1-score   support

           0       0.88      0.92      0.90       513
           1       0.78      0.71      0.74       213

    accuracy                           0.86       726
   macro avg       0.83      0.81      0.82       726
weighted avg       0.85      0.86      0.85       726

[[471  42]
 [ 62 151]]


In [26]:
vgg_model.save('./model_interjection_AUGMENT.keras', overwrite=True)